# Real-Time Transaction Fraud Detection: Exhaustive Domain Feature Engineering Suite

**Dataset:** IEEE-CIS Fraud Detection (590,540 observations, 437 features)  
**Architecture Alignment:** Feast Feature Store + LightGBM + Dynamic Value Cost Router + Exhaustive Feature Suite

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Connect to DuckDB offline store in read_only mode
db_path = 'feature_store.duckdb' if os.path.exists('feature_store.duckdb') else '../feature_store.duckdb'
print(f"Connecting to DuckDB at {db_path}...")
conn = duckdb.connect(db_path, read_only=True)
df_tx = conn.execute('SELECT * FROM transactions').fetchdf()
df_id = conn.execute('SELECT * FROM identities').fetchdf()

print(f"Transactions Shape: {df_tx.shape}")
print(f"Identities Shape: {df_id.shape}")

Connecting to DuckDB at ../feature_store.duckdb...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Transactions Shape: (590540, 394)
Identities Shape: (144233, 41)


In [2]:
# 2. Entity Merge & Identity Coverage Check
df = pd.merge(df_tx, df_id, on='TransactionID', how='left')
id_coverage = (~df['id_01'].isna()).mean() * 100
print(f"Merged Matrix Shape: {df.shape}")
print(f"Identity Metadata Coverage: {id_coverage:.2f}% ({id_coverage:.1f}% transactions have device fingerprint)")

Merged Matrix Shape: (590540, 434)
Identity Metadata Coverage: 24.42% (24.4% transactions have device fingerprint)


In [3]:
# 3. Defensive Memory Optimization (Downcasting)
def downcast_dtypes(df_in):
    float_cols = [c for c in df_in.columns if df_in[c].dtype == 'float64']
    int_cols = [c for c in df_in.columns if df_in[c].dtype in ['int64', 'int32']]
    df_in[float_cols] = df_in[float_cols].astype('float32')
    df_in[int_cols] = df_in[int_cols].astype('int32')
    return df_in

df_opt = downcast_dtypes(df.copy())
print("Memory optimization downcasting complete.")

Memory optimization downcasting complete.


In [4]:
# 4. Data Integrity & Precision Assert Check
orig_nans = df.select_dtypes(include=[np.number]).isna().sum().sum()
opt_nans = df_opt.select_dtypes(include=[np.number]).isna().sum().sum()
assert orig_nans == opt_nans, 'Integrity Check Failed: NaN mismatch after downcasting!'
print(f"Data Integrity Check PASSED! Total numeric nulls perfectly preserved: {opt_nans:,}")
df = df_opt

Data Integrity Check PASSED! Total numeric nulls perfectly preserved: 104,178,894


In [5]:
# 5. Domain Analysis 1: Temporal & Diurnal Fraud Patterns (D9 * 24 and TransactionDT)
# Reference t0: 2017-12-01 00:00:00 UTC (Peak US commerce at 14:00-22:00 UTC / 9AM-5PM EST)
df['hour_d9'] = (df['D9'] * 24).round()
df['hour_dt'] = (df['TransactionDT'] // 3600) % 24
df['day_dt'] = (df['TransactionDT'] // 86400) % 7

hourly_fraud = df.groupby('hour_dt')['isFraud'].agg(['count', 'mean']).rename(columns={'count': 'tx_count', 'mean': 'fraud_rate'})
print('=== Hourly Fraud Rate Summary (UTC Timeline) ===')
print(hourly_fraud)

=== Hourly Fraud Rate Summary (UTC Timeline) ===
         tx_count  fraud_rate
hour_dt                      
0           37795    0.031380
1           32797    0.031314
2           26732    0.037483
3           20802    0.038314
4           14839    0.051890
5            9701    0.070302
6            6007    0.077743
7            3704    0.106102
8            2591    0.093014
9            2479    0.089956
10           3627    0.053212
11           6827    0.038816
12          12451    0.030439
13          20315    0.022889
14          28328    0.024216
15          33859    0.025399
16          38698    0.029511
17          40723    0.031530
18          41639    0.035231
19          42115    0.034738
20          41782    0.034273
21          41641    0.034005
22          41139    0.032694
23          39949    0.036997


In [6]:
# 6. Domain Analysis 2: Foreign Currency & 3-Decimal Places Analysis
def get_decimal_len(series):
    return series.astype(str).str.split('.').str[1].fillna('').str.len()

df['decimal_places'] = get_decimal_len(df['TransactionAmt'])
df['is_foreign_currency'] = (df['decimal_places'] >= 3).astype(int)

foreign_summary = df.groupby('is_foreign_currency').agg(
    total_tx=('isFraud', 'count'),
    fraud_rate=('isFraud', 'mean'),
    missing_addr1_pct=('addr1', lambda x: x.isna().mean() * 100)
)
print('=== Foreign Currency vs Address Missingness ===')
print(foreign_summary)

=== Foreign Currency vs Address Missingness ===
                     total_tx  fraud_rate  missing_addr1_pct
is_foreign_currency                                         
0                      528609    0.025357           1.288665
1                       61931    0.117211          95.096155


In [7]:
# 7. Domain Analysis 3: Dual-Tier Entity Resolution & Collision-Free Hash
# Tier 1: Universal Card Base Key (100% complete)
df['card_base_id'] = (
    df['card1'].astype(str) + '_' +
    df['card2'].fillna(0).astype(int).astype(str) + '_' +
    df['card3'].fillna(0).astype(int).astype(str) + '_' +
    df['card4'].fillna('unk').astype(str) + '_' +
    df['card5'].fillna(0).astype(int).astype(str) + '_' +
    df['card6'].fillna('unk').astype(str)
)

# Tier 2: Strict Cardholder UID (Preventing null address collision)
addr_salt = df['addr1'].fillna('NONE_' + df['TransactionID'].astype(str))
email_salt = df['P_emaildomain'].fillna('NONE_' + df['TransactionID'].astype(str))
df['cardholder_uid'] = df['card_base_id'] + '_' + addr_salt.astype(str) + '_' + email_salt.astype(str)

# Collision Test Assert
null_addr_sample = df[df['addr1'].isna()].head(200)
assert null_addr_sample['cardholder_uid'].nunique() == len(null_addr_sample), 'Collision detected in null address records!'
print(f"Unique Card Base Entities: {df['card_base_id'].nunique():,}")
print(f"Unique Cardholder Strict Entities: {df['cardholder_uid'].nunique():,}")
print('Entity Resolution Collision Test PASSED!')

Unique Card Base Entities: 14,893
Unique Cardholder Strict Entities: 234,720
Entity Resolution Collision Test PASSED!


In [8]:
# 8. Feature Engineering 1: Device, Browser & Screen Fingerprint Extraction
def parse_device_corp(val):
    if pd.isna(val): return 'missing'
    s = str(val).lower()
    if 'sm-' in s or 'samsung' in s: return 'samsung'
    if 'ios' in s or 'iphone' in s or 'ipad' in s or 'macos' in s: return 'apple'
    if 'windows' in s: return 'microsoft'
    if 'huawei' in s or 'ale-' in s or 'pra-' in s: return 'huawei'
    if 'lg' in s or 'nexus' in s or 'pixel' in s: return 'google_lg'
    if 'moto' in s: return 'motorola'
    return 'other'

def parse_browser_corp(val):
    if pd.isna(val): return 'missing'
    s = str(val).lower()
    if 'chrome' in s: return 'chrome'
    if 'safari' in s: return 'safari'
    if 'firefox' in s: return 'firefox'
    if 'edge' in s: return 'edge'
    if 'ie' in s: return 'ie'
    if 'samsung' in s: return 'samsung_browser'
    if 'opera' in s: return 'opera'
    return 'other'

def parse_os_family(val):
    if pd.isna(val): return 'missing'
    s = str(val).lower()
    if 'ios' in s: return 'ios'
    if 'android' in s: return 'android'
    if 'windows' in s: return 'windows'
    if 'mac' in s: return 'mac'
    if 'linux' in s: return 'linux'
    return 'other'

def parse_screen_aspect(val):
    if pd.isna(val) or 'x' not in str(val): return 0.0
    parts = str(val).split('x')
    try: return float(parts[0]) / (float(parts[1]) + 1e-5)
    except: return 0.0

df['device_corp'] = df['DeviceInfo'].apply(parse_device_corp).astype('category')
df['browser_corp'] = df['id_31'].apply(parse_browser_corp).astype('category')
df['os_family'] = df['id_30'].apply(parse_os_family).astype('category')
df['screen_aspect_ratio'] = df['id_33'].apply(parse_screen_aspect).astype('float32')
print("Device & Browser Fingerprint features engineered successfully.")

Device & Browser Fingerprint features engineered successfully.


In [9]:
# 9. Feature Engineering 2: Email Domain Matching & Disposable Flags
df['email_domain_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
df.loc[df['P_emaildomain'].isna() | df['R_emaildomain'].isna(), 'email_domain_match'] = -1

disposable_domains = {'mail.com', 'protonmail.com', 'anonymous.com', 'tutanota.com'}
df['is_disposable_email'] = df['P_emaildomain'].isin(disposable_domains).astype(int)
print("Email Domain Matching features engineered successfully.")

Email Domain Matching features engineered successfully.


In [10]:
# 10. Feature Engineering 3: Frequency Encodings & Card Relative Aggregations
df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'])

# Amount Group Aggregations
card_amt_mean = df.groupby('card_base_id')['TransactionAmt'].transform('mean')
card_amt_std = df.groupby('card_base_id')['TransactionAmt'].transform('std').fillna(1.0)
df['amt_to_mean_card'] = (df['TransactionAmt'] / (card_amt_mean + 1e-5)).astype('float32')
df['amt_to_std_card'] = ((df['TransactionAmt'] - card_amt_mean) / (card_amt_std + 1e-5)).astype('float32')

# Entity Frequency Encodings
df['card_base_id_freq'] = df['card_base_id'].map(df['card_base_id'].value_counts()).fillna(1).astype('float32')
df['cardholder_uid_freq'] = df['cardholder_uid'].map(df['cardholder_uid'].value_counts()).fillna(1).astype('float32')
df['addr1_freq'] = df['addr1'].map(df['addr1'].value_counts(dropna=False)).fillna(1).astype('float32')
df['P_emaildomain_freq'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts(dropna=False)).fillna(1).astype('float32')

# Scaled D-Column Deltas
for d_col in ['D1', 'D2', 'D15']:
    if d_col in df.columns:
        card_d_mean = df.groupby('card_base_id')[d_col].transform('mean')
        df[f'{d_col}_to_mean_card'] = (df[d_col] / (card_d_mean + 1e-5)).fillna(0.0).astype('float32')

print("Frequency encodings and relative D-column deltas computed successfully.")

Frequency encodings and relative D-column deltas computed successfully.


In [11]:
# 11. Feature Engineering 4: Option B (Top-3 Medoids per Block = 33 Features)
v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
v_blocks = {
    'V1_11': [f'V{i}' for i in range(1, 12)],
    'V12_34': [f'V{i}' for i in range(12, 35)],
    'V35_52': [f'V{i}' for i in range(35, 53)],
    'V53_74': [f'V{i}' for i in range(53, 75)],
    'V75_94': [f'V{i}' for i in range(75, 95)],
    'V95_137': [f'V{i}' for i in range(95, 138)],
    'V138_166': [f'V{i}' for i in range(138, 167)],
    'V167_216': [f'V{i}' for i in range(167, 217)],
    'V217_278': [f'V{i}' for i in range(217, 279)],
    'V279_321': [f'V{i}' for i in range(279, 322)],
    'V322_339': [f'V{i}' for i in range(322, 340)],
}

selected_v_cols = []
for block_name, cols in v_blocks.items():
    valid_cols = [c for c in cols if c in df.columns]
    if valid_cols:
        variances = df[valid_cols].var().sort_values(ascending=False)
        top3 = variances.head(3).index.tolist()
        selected_v_cols.extend(top3)

print(f"V-Feature Compression (Option B): Selected {len(selected_v_cols)} medoids across 11 blocks.")
print(f"Selected 33 Medoid Columns: {selected_v_cols}")

V-Feature Compression (Option B): Selected 33 medoids across 11 blocks.
Selected 33 Medoid Columns: ['V11', 'V10', 'V5', 'V30', 'V13', 'V29', 'V38', 'V45', 'V37', 'V56', 'V70', 'V54', 'V78', 'V91', 'V76', 'V127', 'V133', 'V128', 'V160', 'V159', 'V165', 'V203', 'V212', 'V204', 'V264', 'V265', 'V263', 'V307', 'V317', 'V308', 'V332', 'V333', 'V331']


In [12]:
# 12. Fast Out-of-Time (OOT) LightGBM Baseline Validation (with Exhaustive Feature Suite)
train_mask = df['TransactionDT'] < (120 * 86400)
val_mask = df['TransactionDT'] >= (120 * 86400)

base_num_cols = [
    'TransactionAmt', 'log_TransactionAmt', 'amt_to_mean_card', 'amt_to_std_card',
    'is_foreign_currency', 'hour_dt', 'day_dt',
    'email_domain_match', 'is_disposable_email',
    'card_base_id_freq', 'cardholder_uid_freq', 'addr1_freq', 'P_emaildomain_freq',
    'screen_aspect_ratio'
] + [f'{d}_to_mean_card' for d in ['D1', 'D2', 'D15'] if f'{d}_to_mean_card' in df.columns] \
  + [f'C{i}' for i in range(1, 15) if f'C{i}' in df.columns] \
  + selected_v_cols

cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'device_corp', 'browser_corp', 'os_family']
feature_cols = base_num_cols.copy()

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        feature_cols.append(col)

X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, 'isFraud']
X_val, y_val = df.loc[val_mask, feature_cols], df.loc[val_mask, 'isFraud']

print(f"Total Model Features: {len(feature_cols)}")
print(f"Temporal Split - Train: {X_train.shape[0]:,} rows (Days 1-120), Val: {X_val.shape[0]:,} rows (Days 121-183)")

dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

params = {
    'objective': 'binary',
    'metric': ['auc', 'average_precision'],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'scale_pos_weight': 10.0,
    'verbosity': -1,
    'n_jobs': -1,
    'seed': 42
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round=150,
    valid_sets=[dtrain, dval],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)

val_preds = model.predict(X_val)
roc_score = roc_auc_score(y_val, val_preds)
pr_score = average_precision_score(y_val, val_preds)

print('='*50)
print(f"Exhaustive Suite OOT Baseline ROC-AUC: {roc_score:.4f}")
print(f"Exhaustive Suite OOT Baseline PR-AUC: {pr_score:.4f}")
print('='*50)

Total Model Features: 72
Temporal Split - Train: 410,601 rows (Days 1-120), Val: 179,939 rows (Days 121-183)
Exhaustive Suite OOT Baseline ROC-AUC: 0.9004
Exhaustive Suite OOT Baseline PR-AUC: 0.4942


## 13. Feature Store Prototype Conclusion
The full exhaustive domain feature suite (Device/Browser parsers, Email matching, Entity frequency encodings, card-relative D-deltas, and 33 V-medoids) is verified and ready for production Feast feature view definitions in Phase 3.